# run_notebook


In [ ]:
from pathlib import Path
import sys
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'AGENTS.md').is_file())
if str(_root) not in sys.path: sys.path.insert(0, str(_root))
if str(_root / '.runtime') not in sys.path: sys.path.insert(0, str(_root / '.runtime'))
if str(_root / '.runtime') not in sys.path: sys.path.insert(0, str(_root / '.runtime'))
__file__ = str(_root / 'scripts/run_notebook.py')
__package__ = 'scripts'
__name__ = 'scripts.run_notebook'


In [ ]:
"""Execute a native notebook and persist its real outputs while the long cell runs."""
import asyncio
import json
import sys
import time
from pathlib import Path

import nbformat
from nbclient import NotebookClient

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from train import replace_with_retry




In [ ]:
async def main():
    path = ROOT / 'TRAIN_ALL.ipynb'
    notebook = nbformat.read(path, as_version=4)
    client = NotebookClient(notebook, timeout=None, kernel_name='demo', resources={'metadata': {'path': str(ROOT)}})
    done = asyncio.Event()

    def save():
        temporary = path.with_suffix('.ipynb.tmp')
        temporary.write_text(nbformat.writes(notebook), encoding='utf-8')
        replace_with_retry(temporary, path)

    async def on_cell_executed(cell, cell_index, execute_reply):
        save()
        print(f'Cell {cell_index + 1}/{len(notebook.cells)} executed', flush=True)

    client.on_cell_executed = on_cell_executed

    async def persist_outputs():
        while not done.is_set():
            try:
                await asyncio.wait_for(done.wait(), timeout=15)
            except asyncio.TimeoutError:
                save()

    task = asyncio.create_task(persist_outputs())
    try:
        notebook.metadata['execution_status'] = 'running'
        save()
        await client.async_execute()
        notebook.metadata['execution_status'] = 'complete'
    except BaseException as error:
        notebook.metadata['execution_status'] = 'failed'
        notebook.metadata['execution_error'] = str(error)[:3000]
        status_path = ROOT / 'output/training_status.json'
        try:
            status = json.loads(status_path.read_text(encoding='utf-8'))
        except (FileNotFoundError, json.JSONDecodeError):
            status = {'dataset': 'ALL', 'epochs': 100}
        status.update(state='failed', detail=str(error)[:1000])
        temporary = status_path.with_suffix('.tmp')
        temporary.write_text(json.dumps(status), encoding='utf-8')
        replace_with_retry(temporary, status_path)
        raise
    finally:
        done.set()
        await task
        save()


